In [5]:
import mlrun

from dotenv import load_dotenv
# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
load_dotenv() 

import os
# https://docs.mlrun.org/en/stable/store/datastore.html#s3
# print(os.environ['AWS_ACCESS_KEY_ID'])
# print(os.environ['AWS_SECRET_ACCESS_KEY'])
# print(os.environ['MLRUN_AWS_ROLE_ARN'])

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
print(artifact_path)
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

file://c:/Work/Folder_1/Project_folder/LLM_project_3_FineTune_MLOps/Finetune-legal-llm-mlops


In [2]:
project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory

# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

In [3]:
system_prompt = """
You are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement that may be true or false based on the content of the contract. Label with entailment for true, contradidiction for false, and not_mentioned if the hypothesis is cannot be confirmed or denied bcause it is not memtioned in the content of the contract. The hypotheses with their corresponding ids are as follows:

nda-1: All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2: Confidential Information shall only include technical information.
nda-3: Confidential Information may include verbally conveyed information.
nda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement.
nda-5: Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7: Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8: Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regulation or judicial process to disclose any Confidential Information.
nda-10: Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.
nda-11: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
nda-12: Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.
nda-13: Receiving Party may acquire information similar to Confidential Information from a third party.
nda-15: Agreement shall not grant Receiving Party any right to Confidential Information.
nda-16: Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
nda-17: Receiving Party may create a copy of some Confidential Information in some circumstances.
nda-18: Receiving Party shall not solicit some of Disclosing Party's representatives.
nda-19: Some obligations of Agreement may survive termination of Agreement.
nda-20: Confidential Information may include verbally conveyed information.

Read the contract and respond with a structured JSON document that contains a list of JSON objects for each hypothesis that is not ignored. Each object containins the id of the hypothesis, the label (entailment, contradiction, not_mentioned), the hypothesis statement, and the source clause from the contract that supports the analysis for each hypothesis. For hypotheses that are labeled as not_mentioned leave the source clause field blank. Only respond with the JSON document and do not provide any additional information.

The format of the JSON document should be as follows:
[
    {'hypothesis_id': 'nda-11', 'label': 'entailment', 'hypothesis': 'Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.', 'source_clause': 'The Recipient will not copy or reproduce the Confidential Information except as reasonably required for the purposes contemplated in this Agreement, and will ensure that any confidentiality or other proprietary rights notices on the Confidential Information are reproduced on all copies.'},
    {'hypothesis_id': 'nda-16', 'label': 'contradiction', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.', 'source_clause': 'All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: (a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or (b) promptly upon request by the UNHCR at any time.'},
    {'hypothesis_id': 'nda-15', 'label': 'not_mentioned', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.', 'source_clause': ''}
]
"""

In [6]:
prompt_template=[
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": "{contract}",
    }
]

project.log_llm_prompt(
    key="contract_extractor_prompt",
    prompt_template=prompt_template,
    prompt_legend={
        "issue_description": {
            "field": "contract",
            "description": "The legal contract to extract data from",
        },
    },
    invocation_config={
        "temperature": 0.2,
        "top_p": 1.,
        "max_tokens": 1500
    },
    description="Prompt template for ",
    tag=datetime.now().strftime("%Y%m%d_%H%M") # this is the version
)


In [7]:
project.save()

In [10]:
all_prompts = project.list_llm_prompts()

for prompt in all_prompts:
    print(prompt.metadata.key)

contract_extractor_prompt
contract_extractor_prompt


In [18]:
dir(project.list_llm_prompts(name="contract_extractor_prompt", tag="latest")[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply_enrichment_before_to_dict_completion',
 '_default_fields_to_strip',
 '_dict_fields',
 '_enrich_field',
 '_fields_to_enrich',
 '_fields_to_serialize',
 '_fields_to_skip_validation',
 '_get_file_body',
 '_is_valid_field_value_for_serialization',
 '_metadata',
 '_resolve_field_value_by_method',
 '_resolve_initial_to_dict_fields',
 '_resolve_suffix',
 '_resolve_target_hash_path',
 '_serialize_field',
 '_spec',
 '_src_is_temp',
 '_status',
 '_store_prefix',
 '_upload_body',
 '_upload_file',
 '_verify_dict',
 '_verify_list',
 'base_dict',
 'before_log',
 'copy',
 'db_key',
 'export',
 'extra_data',
 'format',
 'from_dic

In [37]:
project.list_llm_prompts(name="contract_extractor_prompt", tag="latest")[0].read_prompt()

[{'role': 'system',
  'content': "\nYou are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement that may be true or false based on the content of the contract. Label with entailment for true, contradidiction for false, and not_mentioned if the hypothesis is cannot be confirmed or denied bcause it is not memtioned in the content of the contract. The hypotheses with their corresponding ids are as follows:\n\nnda-1: All Confidential Information shall be expressly identified by the Disclosing Party.\nnda-2: Confidential Information shall only include technical information.\nnda-3: Confidential Information may include verbally conveyed information.\nnda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement.\nnda-5: Receiving Party may share some Confidential Information with some of Receiving Party's employe